# CDD Notebook 10 — Stage 2: Contrastive PCA Decomposition

This notebook decomposes each causally validated circuit component into contrastive principal directions using paired truthful vs. hallucinated activations.

Expected headline result: mean effective rank = 57.74.

In [ ]:
from sklearn.decomposition import PCA
import numpy as np, torch, gc
from tqdm.auto import tqdm

VARIANCE_THRESHOLD = 0.80
component_activations = {}

model.eval()
with torch.no_grad():
    for pair in tqdm(pairs, desc='Collecting activations for cPCA'):
        tok_t = tokenise(pair['truthful'])
        tok_h = tokenise(pair['hallucinated'])
        _, cache_t = model.run_with_cache(tok_t, names_filter=hook_filter)
        _, cache_h = model.run_with_cache(tok_h, names_filter=hook_filter)

        for _, row in df_circuit.iterrows():
            comp  = row['component']
            layer = int(row['layer'])
            if row['type'] == 'mlp':
                hname = f'blocks.{layer}.hook_mlp_out'
                act_t = cache_t[hname][0, -1].cpu().numpy()
                act_h = cache_h[hname][0, -1].cpu().numpy()
            else:
                head  = int(row['head'])
                hname = f'blocks.{layer}.attn.hook_z'
                act_t = cache_t[hname][0, -1, head].cpu().numpy()
                act_h = cache_h[hname][0, -1, head].cpu().numpy()
            if comp not in component_activations:
                component_activations[comp] = {'truthful': [], 'hallucinated': []}
            component_activations[comp]['truthful'].append(act_t)
            component_activations[comp]['hallucinated'].append(act_h)
        del cache_t, cache_h
        gc.collect()

print(f'Collected activations for {len(component_activations)} circuit components')

In [ ]:
causal_directions = {}
effective_ranks   = {}

for comp, acts in tqdm(component_activations.items(), desc='cPCA'):
    T = np.array(acts['truthful'])
    H = np.array(acts['hallucinated'])
    delta = T - H
    delta_centered = delta - delta.mean(axis=0)
    pca = PCA()
    pca.fit(delta_centered)
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    k = int(np.searchsorted(cumvar, VARIANCE_THRESHOLD)) + 1
    k = max(k, 1)
    causal_directions[comp] = pca.components_[:k]
    effective_ranks[comp]   = k

ranks = list(effective_ranks.values())
print(f'Mean effective rank  : {np.mean(ranks):.2f}')
print(f'Median effective rank: {np.median(ranks):.1f}')
print(f'Max effective rank   : {np.max(ranks)}')
print(f'Components with rank=1: {sum(1 for r in ranks if r==1)}/{len(ranks)}')